# Derivadas, Parciais e Integrais em IA

## Live 2 - Applied Math for Data Science

Nesta aula, vamos conectar tres ideias centrais do calculo com um caso real de desenvolvimento de IA:

- **Derivada**: taxa de mudanca do erro
- **Derivada parcial**: efeito de cada parametro do modelo
- **Integral / acumulacao**: como resumimos comportamento ao longo do tempo ou de uma distribuicao

O estudo de caso sera o treinamento de um modelo de regressao linear com **gradient descent** para prever preco de casas a partir da metragem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)

print('Ambiente pronto para a Live 2.')

---

## 1. Construindo um problema realista

Vamos simular um pequeno dataset de precos de casas. A metragem influencia o preco, mas existe ruido de mercado: localizacao, acabamento, oferta e demanda.

In [ ]:
rng = np.random.default_rng(42)

metros = np.linspace(35, 220, 80)
preco_real = 3200 * metros + 45000
ruido = rng.normal(0, 25000, size=metros.shape)
precos = preco_real + ruido

X = metros.reshape(-1, 1)
y = precos.reshape(-1, 1)

print(f'Temos {len(X)} casas no dataset.')
print(f'Metragem minima: {X.min():.1f} m2')
print(f'Metragem maxima: {X.max():.1f} m2')

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(X, y, alpha=0.75, color='#7bdff2', label='Dados observados')
plt.plot(X, preco_real, color='#ff70a6', linewidth=2.5, label='Tendencia real aproximada')
plt.title('Preco de casas x metragem')
plt.xlabel('Metragem (m2)')
plt.ylabel('Preco (R$)')
plt.legend()
plt.show()

### Modelo escolhido

Vamos usar o modelo mais simples possivel:

$$\hat{y} = wx + b$$

Onde:
- $w$ controla a inclinacao da reta
- $b$ controla o deslocamento vertical

---

## 2. A funcao de custo

Queremos medir o erro medio do modelo. Vamos usar o **Mean Squared Error (MSE)**:

$$J(w, b) = \frac{1}{n} \sum_{i=1}^{n}(\hat{y}_i - y_i)^2$$

Essa funcao depende de **duas variaveis**: $w$ e $b$. Logo, precisamos de **derivadas parciais**.

In [ ]:
def prever(X, w, b):
    return w * X + b

def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

w_inicial = 1000.0
b_inicial = 0.0

pred_inicial = prever(X, w_inicial, b_inicial)
print(f'Erro inicial: {mse(y, pred_inicial):,.2f}')

### Por que derivadas entram aqui?

A derivada indica a taxa de mudanca do erro.

As derivadas **parciais** nos dizem:
- como o erro muda se alterarmos apenas $w$
- como o erro muda se alterarmos apenas $b$

Essas duas informacoes sao exatamente o que precisamos para treinar o modelo.

---

## 3. Derivadas parciais da loss

Para o MSE da regressao linear, temos:

$$\frac{\partial J}{\partial w} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)x_i$$

$$\frac{\partial J}{\partial b} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)$$

Essas expressoes dizem o quanto devemos corrigir cada parametro.

In [ ]:
def gradientes(X, y, w, b):
    n = len(X)
    y_pred = prever(X, w, b)
    erro = y_pred - y
    dJ_dw = (2 / n) * np.sum(erro * X)
    dJ_db = (2 / n) * np.sum(erro)
    return dJ_dw, dJ_db

dJ_dw, dJ_db = gradientes(X, y, w_inicial, b_inicial)
print(f'dJ/dw = {dJ_dw:,.2f}')
print(f'dJ/db = {dJ_db:,.2f}')

Se o gradiente for muito positivo, significa que aumentar o parametro piora o erro. Se for negativo, aumentar o parametro pode ajudar.

---

## 4. Gradient Descent em acao

A regra de atualizacao e:

$$w \leftarrow w - \eta \frac{\partial J}{\partial w}$$
$$b \leftarrow b - \eta \frac{\partial J}{\partial b}$$

Onde $\eta$ e a taxa de aprendizado.

In [ ]:
X_norm = (X - X.mean()) / X.std()
y_norm = (y - y.mean()) / y.std()

w = 0.0
b = 0.0
lr = 0.08
epocas = 250

historico_loss = []
historico_w = []
historico_b = []

for epoca in range(epocas):
    dJ_dw, dJ_db = gradientes(X_norm, y_norm, w, b)
    w = w - lr * dJ_dw
    b = b - lr * dJ_db
    loss = mse(y_norm, prever(X_norm, w, b))
    historico_loss.append(loss)
    historico_w.append(w)
    historico_b.append(b)

print(f'w final: {w:.4f}')
print(f'b final: {b:.4f}')
print(f'loss final: {historico_loss[-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(historico_loss, color='#ffd166', linewidth=2.5)
axes[0].set_title('Queda da loss ao longo das epocas')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('MSE normalizado')

axes[1].plot(historico_w, label='w', color='#7bdff2', linewidth=2)
axes[1].plot(historico_b, label='b', color='#ff70a6', linewidth=2)
axes[1].set_title('Atualizacao dos parametros')
axes[1].set_xlabel('Epoca')
axes[1].legend()

plt.tight_layout()
plt.show()

### Interpretacao

- A **derivada** informa a inclinacao local da loss
- As **derivadas parciais** dizem como corrigir cada parametro
- O treinamento funciona porque repetimos pequenas atualizacoes seguindo o gradiente

---

## 5. Resultado final do modelo

In [ ]:
x_media = X.mean()
x_std = X.std()
y_media = y.mean()
y_std = y.std()

pred_norm = prever(X_norm, w, b)
pred_reais = pred_norm * y_std + y_media

plt.figure(figsize=(10, 5))
plt.scatter(X, y, alpha=0.65, color='#7bdff2', label='Dados')
plt.plot(X, pred_reais, color='#ff70a6', linewidth=3, label='Reta aprendida')
plt.title('Modelo ajustado por gradient descent')
plt.xlabel('Metragem (m2)')
plt.ylabel('Preco (R$)')
plt.legend()
plt.show()

Esse e um exemplo simples, mas a mesma logica escala para redes neurais profundas: milhares ou milhoes de parametros, cada um com sua derivada parcial.

---

## 6. Onde a integral entra nesse contexto?

Mesmo quando o treinamento depende mais diretamente de derivadas e parciais, a ideia de **integral como acumulacao** continua aparecendo em IA.

Alguns exemplos:
- acumular erro ao longo do tempo em series temporais
- calcular energia de um sinal
- integrar uma densidade para obter probabilidade
- resumir desempenho ao longo de todos os limiares

In [ ]:
x = np.linspace(-3, 3, 400)
densidade = (1 / np.sqrt(2 * np.pi)) * np.exp(-(x ** 2) / 2)

mascara = (x >= -1) & (x <= 1)
prob_intervalo = np.trapezoid(densidade[mascara], x[mascara])

plt.figure(figsize=(10, 5))
plt.plot(x, densidade, color='#7bdff2', linewidth=2.5)
plt.fill_between(x[mascara], densidade[mascara], color='#ffd166', alpha=0.45)
plt.title('Integral de uma densidade: probabilidade em um intervalo')
plt.xlabel('z')
plt.ylabel('f(z)')
plt.show()

print(f'Probabilidade aproximada de Z estar entre -1 e 1: {prob_intervalo:.4f}')

Aqui usamos a **regra do trapezio** para aproximar a integral numericamente. Em IA, isso aparece em analise probabilistica, inferencia e avaliacao de modelos.

---

## 7. Conclusoes

Voce viu um ciclo completo:

1. Definimos um problema real de predicao
2. Escolhemos uma funcao de custo
3. Calculamos derivadas parciais
4. Atualizamos parametros com gradient descent
5. Usamos a ideia de integral para acumulacao e probabilidade

Em resumo:
- **Derivadas** ajudam a detectar mudanca
- **Parciais** ajudam a otimizar modelos com varias variaveis
- **Integrais** ajudam a somar comportamento ao longo de um dominio

Esse trio forma a base matematica de boa parte da IA moderna.